In [3]:
!pip install -q accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.4 MB/s eta 0:00:00:00:0100:01


In [4]:
import pandas as pd
import numpy as np
import random
from tqdm import tqdm
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig

In [5]:
# LOAD MISTRAL-7B LOCALL
print("Loading Mistral-7B-Instruct...")
model_id = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(model_id)

# 1. Create the new Quantization Configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",       # Uses NormalFloat4, which is better for performance
    bnb_4bit_use_double_quant=True   # Saves even more GPU memory
)

# 2. Pass the config into the model loader
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=bnb_config,
)

# Create the text-generation pipeline
pipe = pipeline(
    "text-generation", 
    model=model, 
    tokenizer=tokenizer, 
    max_new_tokens=100,    
    do_sample=True,     
    temperature=0.1,        
    pad_token_id=tokenizer.eos_token_id,
    return_full_text=False 
)

print("Model loaded successfully!")

Loading Mistral-7B-Instruct...


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'temperature', 'do_sample', 'max_new_tokens', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Model loaded successfully!


In [6]:
# LOAD DATA
file_path = '/kaggle/input/datasets/noor3027/clinical-data/Clinical_and_Other_Features.xlsx'
try:
    df_raw = pd.read_excel(file_path, header=None)
    data = df_raw.iloc[3:].copy()
except FileNotFoundError:
    print("File not found. Using dummy data for testing.")
    data = pd.DataFrame({0: ['Patient_1'], 36: ['L'], 37: ['10 oclock'], 69: ['1'], 70: ['1'], 71: ['2'], 75: ['2.5'], 50: ['1'], 48: ['0'], 51: ['0'], 52: ['0'], 49: ['0'], 72: ['0'], 80: ['0'], 81: ['0']})

In [7]:
# DEFINE MAPPINGS
shape_map = {"0": "oval", "1": "irregular", "2": "lobular", "3": "reniform", "4": "stellate"}
margin_map = {"0": "obscured", "1": "ill-defined", "2": "spiculated", "3": "indistinct", "4": "circumscribed"}
density_map = {"0": "heterogeneous", "1": "scattered", "2": "minimal", "3": "moderate", "4": "extremely", "5": "predominantly fatty"}
echo_map = {"0": "hypoechoic", "1": "hyperechoic", "2": "isoechoic", "3": "anechoic", "4": "irregular", "5": "mixed", "6": "boundary"}
calc_map = {"0": "pleomorphic", "1": "present (unspecified)", "2": "heterogeneous", "3": "microcalcifications", "4": "linear", "5": "clustered", "6": "amorphous", "7": "branching"}
yes_no_map = {"0": "no", "1": "yes"}
side_map = {"L": "left", "R": "right"}

In [8]:
# CLEANING FUNCTION
def clean_val(val, mapping=None):
    if pd.isna(val):
        return None
    val_str = str(val).strip().upper()
    if val_str in ['NC', 'NA', 'NP', 'NAN', 'NONE', '']:
        return None
    
    clean_key = str(val).strip()
    if clean_key.endswith('.0'):
        clean_key = clean_key[:-2]
        
    if mapping:
        return mapping.get(clean_key, clean_key) # fallback to key if not in map
    return clean_key

In [9]:
# EXTRACT DATA COLUMNS
df = pd.DataFrame({
    'patient_id': data[0],
    'laterality': data[36].apply(lambda x: clean_val(x, side_map)), 
    'position': data[37].apply(lambda x: clean_val(x)),
    'density': data[69].apply(lambda x: clean_val(x, density_map)),
    
    # Mammography Features
    'm_shape': data[70].apply(lambda x: clean_val(x, shape_map)),
    'm_margin': data[71].apply(lambda x: clean_val(x, margin_map)),
    'm_size': data[75].apply(lambda x: clean_val(x)),
    'arch_dist': data[72].apply(lambda x: clean_val(x, yes_no_map)), # Architectural distortion
    'calcifications': data[74].apply(lambda x: clean_val(x, calc_map)), # Calcifications
    
    # Ultrasound Features
    'us_shape': data[76].apply(lambda x: clean_val(x, shape_map)),
    'us_margin': data[77].apply(lambda x: clean_val(x, margin_map)),
    'us_size': data[78].apply(lambda x: clean_val(x)),
    'us_echo': data[79].apply(lambda x: clean_val(x, echo_map)),
    'us_solid': data[80].apply(lambda x: clean_val(x, yes_no_map)), # Solid mass
    'us_shadow': data[81].apply(lambda x: clean_val(x, yes_no_map)), # Posterior shadowing
    
    # Associated/MRI Findings
    'nodes': data[50].apply(lambda x: clean_val(x, yes_no_map)),
    'multifocal': data[48].apply(lambda x: clean_val(x, yes_no_map)),
    'skin_inv': data[51].apply(lambda x: clean_val(x, yes_no_map)),
    'pec_inv': data[52].apply(lambda x: clean_val(x, yes_no_map)),
    'contralateral': data[49].apply(lambda x: clean_val(x, yes_no_map)) # Contralateral involvement
})

In [22]:
# MISTRAL-SPECIFIC PROMPT GENERATOR
def create_medical_prompt(row):
    findings = []
    
    if row['density']: findings.append(f"Breast density: {row['density']}.")
    
    # Location
    loc_str = []
    if row['laterality']: 
        loc_str.append(f"{row['laterality']} breast")
    
    if row['position']: 
        pos_raw = str(row['position']).strip().upper()
        
        # If the cell is literally just 'L' or 'R', ignore it (we already have laterality)
        if pos_raw in ['L', 'R']:
            pos_clean = ""
        else:
            # Strip prefixes and format nicely
            pos_clean = pos_raw.replace('L ', '').replace('R ', '').strip()
            
            # Catch plain numbers (e.g., '1' -> "1 o'clock")
            if pos_clean.isdigit():
                pos_clean = f"{pos_clean} o'clock"
            # Catch things like '9 MEDIAL' -> "9 o'clock medial"
            elif pos_clean[0].isdigit() and ' ' in pos_clean:
                parts = pos_clean.split(' ', 1)
                pos_clean = f"{parts[0]} o'clock {parts[1].lower()}"
                
        if pos_clean:
            loc_str.append(f"{pos_clean} position")
            
    if loc_str: findings.append(f"Mass location: {' at '.join(loc_str)}.")
    
    shape = row['m_shape'] or row['us_shape'] 
    if shape: findings.append(f"Lesion shape: {shape}.")
        
    margin = row['m_margin'] or row['us_margin']
    if margin: findings.append(f"Margins: {margin}.")
        
    size = row['m_size'] or row['us_size']
    if size: findings.append(f"Maximum dimension: {size} cm.")
    
    if row['us_echo']: findings.append(f"Echogenicity: {row['us_echo']}.")
    if row['us_solid'] == 'yes': findings.append("Lesion appears solid on ultrasound.")
    if row['us_shadow'] == 'yes': findings.append("Posterior acoustic shadowing is noted.")
    if row['arch_dist'] == 'yes': findings.append("Associated architectural distortion is present.")
    if row['calcifications']: findings.append(f"Associated calcifications: {row['calcifications']}.")
    
    if row['nodes'] == 'yes': findings.append("Suspicious axillary lymphadenopathy present.")
    if row['multifocal'] == 'yes': findings.append("Multicentric/multifocal disease pattern.")
    if row['skin_inv'] == 'yes': findings.append("Skin or nipple involvement noted.")
    if row['pec_inv'] == 'yes': findings.append("Pectoral muscle involvement noted.")
    if row['contralateral'] == 'yes': findings.append("Suspicious findings noted in the contralateral breast.")
    
    findings_str = " ".join(findings)
    if not findings_str: return None
    
    style_hints = [
        "Begin by describing the background breast tissue density before detailing the primary lesion.",
        "Start directly with the primary mass, mentioning background density later.",
        "Use a highly concise, telegraphic medical style.",
        "Use a fluid, narrative clinical style.",
        "Integrate the associated findings into the same sentence as the primary lesion description.",
        "Write in short, punchy, objective declarative sentences."
    ]
    chosen_style = random.choice(style_hints)
    
    # Notice the [INST] and [/INST] tags specifically for Mistral
    messages = [
        {"role": "system", "content": f"You are a strict medical data parser. Convert the descriptors into a short, fluid radiology sentence.\nSTYLE: {chosen_style}\nRULES: Use ONLY provided data. Do not invent features, anatomy, or sizes. Do not mention missing data or use phrases like 'no additional descriptors provided'. Output ONLY the clinical text."},
        
        # Example 1: Rich Data
        {"role": "user", "content": "Descriptors: Breast density: scattered. Mass location: Left breast at 2 o'clock position. Lesion shape: irregular. Margins: spiculated. Maximum dimension: 1.5 cm. Associated calcifications: pleomorphic."},
        {"role": "assistant", "content": "The background breast parenchyma exhibits scattered fibroglandular density. An irregular mass measuring 1.5 cm with spiculated margins is identified in the left breast at the 2 o'clock position, associated with pleomorphic calcifications."},
        
        # Example 2: Medium Data
        {"role": "user", "content": "Descriptors: Mass location: Right breast at 10 o'clock position. Suspicious axillary lymphadenopathy present."},
        {"role": "assistant", "content": "A mass is identified in the right breast at the 10 o'clock position. Suspicious ipsilateral axillary lymphadenopathy is noted."},
        
        # Example 3: VERY Sparse Data (This teaches it NOT to say "No data provided")
        {"role": "user", "content": "Descriptors: Mass location: Left breast."},
        {"role": "assistant", "content": "An abnormality is identified within the left breast."},
        
        # The Actual Patient Data
        {"role": "user", "content": f"Descriptors: {findings_str}\nReport:"}
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return prompt

In [23]:
# MISTRAL INFERENCE FUNCTION
def get_llm_report(prompt):
    if not prompt: return "No descriptive imaging data available for this patient."
    try:
        outputs = pipe(prompt)
        report = outputs[0]['generated_text'].strip()
        
        # POST-PROCESSING TO CHOP OFF CHATBOT NOTES
        # 1. Split by double newline and take only the first paragraph
        report = report.split('\n\n')[0].strip()
        # 2. Split by '(' in case it started a note like "(Note: ...)" on the same line
        report = report.split('(')[0].strip()
        # 3. Split by "Note:" just to be extra safe
        report = report.split('Note:')[0].strip()
        
        return report
    except Exception as e:
        print(f"\n⚠️ Inference Error: {e}")
        return "Error generating report."

In [24]:
# RUN TEST
print("Starting report generation for test set...")
test_df = df.head(5).copy()
tqdm.pandas()
test_df['generated_report'] = test_df.progress_apply(lambda row: get_llm_report(create_medical_prompt(row)), axis=1)

print("\n" + "="*50)
print("🩺 TEST RESULTS: GENERATED RADIOLOGY REPORTS")
print("="*50)
for index, row in test_df.iterrows():
    print(f"Patient ID: {row['patient_id']}")
    print(f"Report: {row['generated_report']}")
    print("-" * 50)

Starting report generation for test set...


100%|██████████| 5/5 [00:24<00:00,  4.81s/it]


🩺 TEST RESULTS: GENERATED RADIOLOGY REPORTS
Patient ID: Breast_MRI_001
Report: A mass is detected in the left breast at the 9 o'clock medial position.
--------------------------------------------------
Patient ID: Breast_MRI_002
Report: A mass is detected in the left breast at the 1 o'clock position.
--------------------------------------------------
Patient ID: Breast_MRI_003
Report: The background breast parenchyma demonstrates scattered fibroglandular density. A mass with associated pleomorphic calcifications is identified at the 2 o'clock position. Suspicious axillary lymphadenopathy is present, and a multicentric/multifocal disease pattern is noted.
--------------------------------------------------
Patient ID: Breast_MRI_004
Report: An abnormality is identified in the left breast.
--------------------------------------------------
Patient ID: Breast_MRI_005
Report: A mass is detected in the right breast at the 3 o'clock position, with the presence of suspicious axillary lymphade

In [26]:
# RUN GENERATION
print("Starting ful report generation...")
tqdm.pandas()

df['generated_report'] = df.progress_apply(lambda row: get_llm_report(create_medical_prompt(row)), axis=1)

Starting ful report generation...


100%|██████████| 922/922 [1:11:14<00:00,  4.64s/it]


In [27]:
# SAVE TO CSV
output_df = df[['patient_id', 'generated_report']]
output_df.to_csv('patient_radiology_reports.csv', index=False)

print("Success! Generated reports saved to 'patient_radiology_reports.csv'")
output_df.head()

Success! Generated reports saved to 'patient_radiology_reports.csv'


,patient_id,generated_report
3,Breast_MRI_001,A mass is detected in the left breast at the 9...
4,Breast_MRI_002,A mass is detected in the left breast at the 1...
5,Breast_MRI_003,The background breast parenchyma demonstrates ...
6,Breast_MRI_004,An abnormality is present in the left breast.
7,Breast_MRI_005,A mass is detected in the right breast at the ...
